# Lab 23 — Non-LOCO 80/20 vs Nested LOCO-Optuna

Mục tiêu: đánh giá cùng bộ dữ liệu 920 ca UCI bằng cách chia stratified 80/20, giữ nguyên P1 sentinel-aware, hai model và không gian Optuna của Lab 22.

- `N_REPEATS = 5` giúp giảm phụ thuộc vào một lần chia dữ liệu.
- Đặt `N_REPEATS = 1` và seed `42` để tái hiện cách 80/20 cũ.
- Kết quả non-LOCO được so sánh với số liệu Lab 22 đã báo cáo.
- Không dùng `site` để chia train/test; vì vậy đây là đánh giá IID/internal validation, không phải kiểm tra bệnh viện chưa thấy.

In [ ]:
!pip -q install optuna lightgbm seaborn

import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42
N_REPEATS = 5
RUN_SEEDS = [42, 52, 62, 72, 82][:N_REPEATS]
N_TRIALS = 30
TEST_SIZE = 0.20
THRESHOLD = 0.50
OUTPUT_DIR = Path('/content/uci_multicenter_non_loco_80_20_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland':'processed.cleveland.data', 'hungarian':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data', 'va':'processed.va.data'}
COLUMNS = FEATURES + ['num']

def read_uci(site, filename):
    frame = pd.read_csv(f'{BASE_URL}/{filename}', names=COLUMNS, na_values=['?'],
                        skipinitialspace=True).apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat([read_uci(site, filename) for site, filename in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
display(data.groupby('site')[TARGET].agg(['size','sum','mean']).round(4))
print('Validation:', f'{1 - TEST_SIZE:.0%}/{TEST_SIZE:.0%} stratified random split',
      '| repeats:', N_REPEATS, '| Optuna trials per model per repeat:', N_TRIALS)

In [ ]:
FIXED_PARAMS = {
    'Logistic Regression': {'max_iter': 2000},
    'LightGBM': {'n_estimators': 250, 'learning_rate': 0.03, 'num_leaves': 15,
        'min_child_samples': 15, 'subsample': 0.9, 'colsample_bytree': 0.9, 'reg_lambda': 1.0},
}

def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def build_pipeline(model_name, params=None, seed=RANDOM_STATE):
    params = dict(params or {})
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                       ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    preprocessor = ColumnTransformer([('numeric', numeric, NUMERICAL_FEATURES),
                                     ('categorical', categorical, CATEGORICAL_FEATURES)])
    if model_name == 'Logistic Regression':
        estimator = LogisticRegression(random_state=seed, **params)
    else:
        estimator = LGBMClassifier(random_state=seed, verbosity=-1, **params)
    return Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])

def prepare(frame):
    ready = apply_p1(frame)
    return ready[FEATURES], ready[TARGET]

def score_probability(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan,
        'brier': brier_score_loss(y_true, probability),
        'false_negatives': int(fn), 'false_positives': int(fp)}

In [ ]:
def suggest_params(trial, model_name):
    if model_name == 'Logistic Regression':
        return {
            'C': trial.suggest_float('C', 0.01, 10.0, log=True),
            'solver': trial.suggest_categorical('solver', ['lbfgs', 'liblinear']),
            'max_iter': 2000,
        }
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 7, 63),
        'max_depth': trial.suggest_categorical('max_depth', [-1, 3, 5, 8]),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.70, 1.00),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.70, 1.00),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 5.0),
    }

def tune_on_training(train_frame, model_name, seed):
    y = train_frame[TARGET].to_numpy()
    splits = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=seed).split(
        np.zeros(len(y)), y))

    def objective(trial):
        params = suggest_params(trial, model_name)
        fold_scores = []
        for inner_train_idx, inner_valid_idx in splits:
            inner_train = train_frame.iloc[inner_train_idx]
            inner_valid = train_frame.iloc[inner_valid_idx]
            X_fit, y_fit = prepare(inner_train)
            X_valid, y_valid = prepare(inner_valid)
            model = build_pipeline(model_name, params, seed=seed)
            model.fit(X_fit, y_fit)
            probability = model.predict_proba(X_valid)[:, 1]
            fold_scores.append(roc_auc_score(y_valid, probability))
        return float(np.mean(fold_scores))

    study = optuna.create_study(direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
    started = time.perf_counter()
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=False)
    return study, time.perf_counter() - started

In [ ]:
records = []
best_param_records = []
trial_records = []
for seed in RUN_SEEDS:
    train_idx, test_idx = train_test_split(
        np.arange(len(data)), test_size=TEST_SIZE, stratify=data[TARGET],
        random_state=seed)
    train_frame = data.iloc[train_idx].reset_index(drop=True)
    test_frame = data.iloc[test_idx].reset_index(drop=True)
    X_train, y_train = prepare(train_frame)
    X_test, y_test = prepare(test_frame)
    for model_name in ['Logistic Regression', 'LightGBM']:
        fixed_model = build_pipeline(model_name, FIXED_PARAMS[model_name], seed=seed)
        started = time.perf_counter()
        fixed_model.fit(X_train, y_train)
        fixed_fit_seconds = time.perf_counter() - started
        fixed_probability = fixed_model.predict_proba(X_test)[:, 1]
        records.append({'validation': 'non_LOCO_80_20', 'split_seed': seed,
            'configuration': 'F0_P1_fixed', 'model': model_name,
            'train_rows': len(train_frame), 'test_rows': len(test_frame),
            'tuning_seconds': 0.0, 'fit_seconds': fixed_fit_seconds,
            'best_inner_roc_auc': np.nan, 'best_params': '{}',
            **score_probability(y_test, fixed_probability)})

        study, tuning_seconds = tune_on_training(train_frame, model_name, seed)
        best_params = study.best_trial.params.copy()
        if model_name == 'Logistic Regression':
            best_params['max_iter'] = 2000
        best_param_records.append({'split_seed': seed, 'model': model_name,
            'best_inner_roc_auc': study.best_value,
            'best_params': json.dumps(best_params, sort_keys=True),
            'tuning_seconds': tuning_seconds, 'n_trials': len(study.trials)})
        for trial in study.trials:
            trial_records.append({'split_seed': seed, 'model': model_name,
                'trial_number': trial.number, 'state': str(trial.state),
                'value': trial.value,
                'params': json.dumps(trial.params, sort_keys=True)})

        tuned_model = build_pipeline(model_name, best_params, seed=seed)
        started = time.perf_counter()
        tuned_model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - started
        tuned_probability = tuned_model.predict_proba(X_test)[:, 1]
        records.append({'validation': 'non_LOCO_80_20', 'split_seed': seed,
            'configuration': 'F0_P1_optuna_nested', 'model': model_name,
            'train_rows': len(train_frame), 'test_rows': len(test_frame),
            'tuning_seconds': tuning_seconds, 'fit_seconds': fit_seconds,
            'best_inner_roc_auc': study.best_value,
            'best_params': json.dumps(best_params, sort_keys=True),
            **score_probability(y_test, tuned_probability)})
    print('Completed 80/20 split:', seed)

results_df = pd.DataFrame(records)
best_params_df = pd.DataFrame(best_param_records)
trials_df = pd.DataFrame(trial_records)
display(results_df.round(4))
display(best_params_df)

In [ ]:
summary = results_df.groupby(['configuration', 'model']).agg(
    runs=('split_seed', 'nunique'), test_rows_mean=('test_rows', 'mean'),
    roc_auc_mean=('roc_auc', 'mean'), roc_auc_std=('roc_auc', 'std'),
    roc_auc_worst=('roc_auc', 'min'), pr_auc_mean=('pr_auc', 'mean'),
    recall_mean=('recall', 'mean'), recall_std=('recall', 'std'),
    recall_worst=('recall', 'min'), specificity_mean=('specificity', 'mean'),
    f1_mean=('f1', 'mean'), brier_mean=('brier', 'mean'),
    false_negatives_mean_per_run=('false_negatives', 'mean'),
    false_positives_mean_per_run=('false_positives', 'mean'),
    false_negatives_total=('false_negatives', 'sum'),
    false_positives_total=('false_positives', 'sum'),
    best_inner_roc_auc_mean=('best_inner_roc_auc', 'mean'),
    tuning_seconds_mean=('tuning_seconds', 'mean'),
    fit_seconds_mean=('fit_seconds', 'mean')).reset_index()

baseline = summary[summary['configuration'] == 'F0_P1_fixed'].set_index('model')
delta = summary.copy()
for metric in ['roc_auc_mean', 'roc_auc_worst', 'pr_auc_mean', 'recall_mean',
                'recall_worst', 'specificity_mean', 'brier_mean',
                'false_negatives_total', 'false_positives_total']:
    delta[f'delta_vs_fixed_{metric}'] = delta.apply(
        lambda row: row[metric] - baseline.loc[row['model'], metric], axis=1)

display(summary.sort_values(['model', 'roc_auc_mean'], ascending=[True, False]).round(6))
display(delta.round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=summary, x='model', y='roc_auc_mean', hue='configuration', ax=axes[0])
axes[0].set_title('Non-LOCO 80/20 mean ROC-AUC')
sns.barplot(data=summary, x='model', y='recall_mean', hue='configuration', ax=axes[1])
axes[1].set_title('Non-LOCO 80/20 mean recall at threshold 0.50')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'non_loco_80_20_summary.png', dpi=180, bbox_inches='tight')
plt.show()

## So sánh trực tiếp với Lab 22

Bảng dưới đây dùng các số liệu Lab 22 đã chạy. Nếu có file `nested_optuna_summary.csv` cuối cùng, có thể thay bảng tham chiếu này bằng file đó.

Lưu ý: `roc_auc_worst` của non-LOCO là lần chia 80/20 xấu nhất; `roc_auc_worst` của LOCO là bệnh viện xấu nhất. Hai đại lượng này không hoàn toàn cùng ý nghĩa, nhưng giúp minh họa vì sao LOCO thường khắt khe hơn.

In [ ]:
LOCO_REFERENCE = pd.DataFrame([
    {'validation': 'LOCO', 'configuration': 'F0_P1_fixed', 'model': 'LightGBM',
     'roc_auc_mean': 0.781092, 'roc_auc_std': 0.084134, 'roc_auc_worst': 0.684564,
     'pr_auc_mean': 0.838941, 'recall_mean': 0.786385, 'recall_std': 0.068042,
     'recall_worst': 0.711409, 'specificity_mean': 0.662842, 'f1_mean': 0.775555,
     'brier_mean': 0.187057, 'false_negatives_mean_per_run': 27.50,
     'false_positives_mean_per_run': 31.25},
    {'validation': 'LOCO', 'configuration': 'F0_P1_optuna_nested', 'model': 'LightGBM',
     'roc_auc_mean': 0.799137, 'roc_auc_std': 0.077517, 'roc_auc_worst': 0.716147,
     'pr_auc_mean': 0.862126, 'recall_mean': 0.809457, 'recall_std': 0.105424,
     'recall_worst': 0.711409, 'specificity_mean': 0.584024, 'f1_mean': 0.758738,
     'brier_mean': 0.183609, 'false_negatives_mean_per_run': 24.50,
     'false_positives_mean_per_run': 43.75},
    {'validation': 'LOCO', 'configuration': 'F0_P1_fixed', 'model': 'Logistic Regression',
     'roc_auc_mean': 0.803802, 'roc_auc_std': 0.098955, 'roc_auc_worst': 0.712725,
     'pr_auc_mean': 0.890279, 'recall_mean': 0.781991, 'recall_std': 0.072412,
     'recall_worst': 0.676259, 'specificity_mean': 0.682130, 'f1_mean': 0.807953,
     'brier_mean': 0.153930, 'false_negatives_mean_per_run': 28.25,
     'false_positives_mean_per_run': 19.25},
    {'validation': 'LOCO', 'configuration': 'F0_P1_optuna_nested', 'model': 'Logistic Regression',
     'roc_auc_mean': 0.808415, 'roc_auc_std': 0.079563, 'roc_auc_worst': 0.729964,
     'pr_auc_mean': 0.887319, 'recall_mean': 0.815676, 'recall_std': 0.055321,
     'recall_worst': 0.769784, 'specificity_mean': 0.615126, 'f1_mean': 0.791102,
     'brier_mean': 0.165409, 'false_negatives_mean_per_run': 23.75,
     'false_positives_mean_per_run': 31.75},
])

NON_LOCO_COLUMNS = ['configuration', 'model', 'roc_auc_mean', 'roc_auc_std',
    'roc_auc_worst', 'pr_auc_mean', 'recall_mean', 'recall_std', 'recall_worst',
    'specificity_mean', 'f1_mean', 'brier_mean', 'false_negatives_mean_per_run',
    'false_positives_mean_per_run']
NON_LOCO_REFERENCE = summary[NON_LOCO_COLUMNS].copy()
comparison = pd.concat([LOCO_REFERENCE, NON_LOCO_REFERENCE.assign(validation='non_LOCO_80_20')],
                       ignore_index=True)
display(comparison.sort_values(['model', 'configuration', 'validation']).round(6))

comparison.to_csv(OUTPUT_DIR / 'loco_vs_non_loco_comparison.csv', index=False)

## Cách diễn giải để báo cáo

- Nếu non-LOCO cao hơn LOCO, đó là bằng chứng dữ liệu IID dễ hơn bối cảnh chuyển sang bệnh viện mới; không nên kết luận model tốt hơn chỉ dựa trên 80/20.
- LOCO trả lời câu hỏi robustness multicenter: train trên các bệnh viện còn lại, test trên bệnh viện hoàn toàn chưa thấy.
- Non-LOCO 80/20 phù hợp cho phát triển, sàng lọc nhanh và so sánh hyperparameter; nó không kiểm tra được hospital shift.
- Kết quả chính của Lab 22: nested Optuna cải thiện ROC-AUC, worst-cohort AUC, recall và giảm false negatives; Logistic Regression Optuna là ứng viên ranking tốt nhất.
- Vì specificity giảm và Brier của Logistic Regression Optuna xấu hơn baseline, cần Lab tiếp theo để calibration và chọn threshold theo mục tiêu recall trước khi kết luận về vận hành lâm sàng.

In [ ]:
results_path = OUTPUT_DIR / 'non_loco_80_20_results.csv'
summary_path = OUTPUT_DIR / 'non_loco_80_20_summary.csv'
params_path = OUTPUT_DIR / 'best_params_by_80_20_split.csv'
trials_path = OUTPUT_DIR / 'optuna_trial_history.csv'
delta_path = OUTPUT_DIR / 'non_loco_80_20_delta_vs_fixed.csv'
results_df.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)
best_params_df.to_csv(params_path, index=False)
trials_df.to_csv(trials_path, index=False)
delta.to_csv(delta_path, index=False)
run_config = {'dataset_rows': 920, 'validation': 'repeated stratified 80/20',
    'repeats': N_REPEATS, 'seeds': RUN_SEEDS, 'test_size': TEST_SIZE,
    'preprocessing': 'P1_sentinel_aware', 'threshold': THRESHOLD,
    'n_trials_per_model_per_repeat': N_TRIALS,
    'optimization_metric': 'inner mean ROC-AUC',
    'inner_validation': '3-fold StratifiedKFold on training partition',
    'tuned_models': ['Logistic Regression', 'LightGBM'],
    'feature_engineering': 'none beyond P1', 'site_used_for_split': False,
    'threshold_tuning': 'deferred; fixed at 0.50 for comparison'}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_non_loco_80_20_results', 'zip', OUTPUT_DIR)
print('Saved:', OUTPUT_DIR, zip_path)